# RetroScope REST API demo

This notebook shows the basic workflow with the RetroScope REST API:

1. List gallery captures.
2. Pick the first single-image capture (type == "snapshot").
3. Download that capture.
4. Convert it to grayscale with OpenCV.

RetroScope app needs to run first. By default the API is available on port `8765`.

In [ ]:
from io import BytesIO
from pathlib import Path

import cv2
import httpx
import numpy as np
from IPython.display import display
from PIL import Image

# Use http://<ip-of-pi>:8765 when calling the API from another machine.
API_BASE = "http://127.0.0.1:8765"
client = httpx.Client(base_url=API_BASE, timeout=30.0)

## List all captures

In [ ]:
response = client.get("/api/v1/captures", params={"type": "all", "limit": 500})
response.raise_for_status()

payload = response.json()
captures = payload["captures"]

print(f"Total captures by API: {payload['total']}")
print(f"Captures loaded: {len(captures)}")

# Print a overview of the first few captures.
for item in captures[:10]:
    print(
        item["id"][:12],
        item["type"],
        item["filename"],
        item["captured_at"],
        f"{item['width']}x{item['height']}",
    )

## Select the first single-image capture

The API uses the type `snapshot` for single-image captures.

In [ ]:
single_images = [item for item in captures if item["type"] == "snapshot"]

if not single_images:
    raise RuntimeError("No snapshot captures found in the gallery.")

capture = single_images[0]
capture

## Download and decode the capture

In [ ]:
download_response = client.get(f"/api/v1/captures/{capture['id']}/download")
download_response.raise_for_status()

image_bytes = download_response.content
print(f"Downloaded {len(image_bytes):,} bytes from {capture['filename']}")

# Use the first plane of OME-TIFF.
pil_image = Image.open(BytesIO(image_bytes))
pil_image.seek(0)
rgb = np.asarray(pil_image.convert("RGB"))

print(f"Decoded RGB image shape: {rgb.shape}")

## Convert to grayscale with OpenCV

In [ ]:
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

print(f"Grayscale image shape: {gray.shape}")
print(f"Pixel range: min={gray.min()}, max={gray.max()}")

## Preview original and grayscale output

In [ ]:
rgb_preview = Image.fromarray(rgb)
gray_preview = Image.fromarray(gray).convert("RGB")

preview = Image.new(
    "RGB",
    (rgb_preview.width + gray_preview.width, max(rgb_preview.height, gray_preview.height)),
    "white",
)
preview.paste(rgb_preview, (0, 0))
preview.paste(gray_preview, (rgb_preview.width, 0))

display(preview)